# 07. Microcase: A/B Testing (Email Campaigns)

This is an independent microcase, using the Hillstrom MineThatData dataset, not connected to dunnhumby or PostgreSQL. It answers: did either of the
two email campaigns (Mens, Womens) produce a statistically significant
effect on conversion or spend, compared to the control group (No E-Mail)?

This yields 4 hypothesis tests (2 treatment groups × 2 metrics) answering
one question, so the p-values are corrected for multiple comparisons.
**Bonferroni** is used rather than Benjamini-Hochberg: with only 4 tests,
Benjamini-Hochberg's main advantage, retaining statistical power across
many comparisons, barely applies, and Bonferroni's guarantee (it controls
the chance of any false positive among the 4) is simpler to state.
(`05_campaigns_and_coupons.ipynb` uses Holm for its 30 per-campaign tests.)
Every difference is also reported with a 95% confidence interval, so the
size of each effect is visible, not only whether it is significant.

Two different statistical tests are used, matched to what each metric
actually is:
- **Conversion** (binary: did the customer make a purchase or not) is
  tested with a two-proportion z-test.
- **Spend** (continuous dollar amount, zero for non-converters) is tested
  with a two-sample t-test.

## 1. Setup and Group Sizes

Before testing anything, we inspect the file itself, the same standard applied to every dataset in this project: columns, dtypes, nulls, and a sample of rows, rather than assuming `segment`/`conversion`/`spend` are the only columns or that they hold clean values.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest
from statsmodels.stats.multitest import multipletests

RAW_DATA_DIR = Path("../data/raw/Kevin Hillstrom MineThatData E-MailAnalytics")
filepath = RAW_DATA_DIR / "hillstrom.csv"

df = pd.read_csv(filepath)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print("\nDtypes:")
print(df.dtypes)
print("\nNulls per column:")
print(df.isna().sum())
df.head()

Rows: 64000
Columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']

Dtypes:
recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
conversion           int64
spend              float64
dtype: object

Nulls per column:
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [2]:
print(f"conversion values: {sorted(df['conversion'].unique())}")
print(f"Rows with spend < 0: {(df['spend'] < 0).sum()}")
print(f"Rows with conversion == 1 but spend == 0: {((df['conversion'] == 1) & (df['spend'] == 0)).sum()}")
print(f"Rows with conversion == 0 but spend > 0: {((df['conversion'] == 0) & (df['spend'] > 0)).sum()}")

conversion values: [np.int64(0), np.int64(1)]
Rows with spend < 0: 0
Rows with conversion == 1 but spend == 0: 0
Rows with conversion == 0 but spend > 0: 0


`conversion` is confirmed strictly binary, `spend` has no negative values, and `conversion`/`spend` agree with each other (nobody converts with zero spend or spends without converting), so the two metrics tested below are measuring consistent things rather than two independently-computed columns that happen to disagree. With that confirmed, we look at the three experimental groups themselves.

In [3]:
print("Group sizes:")
print(df["segment"].value_counts())

print("\nOutcomes by group (visit rate, conversion rate, mean spend):")
print(df.groupby("segment")[["visit", "conversion", "spend"]].mean())

Group sizes:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Outcomes by group (visit rate, conversion rate, mean spend):
                  visit  conversion     spend
segment                                      
Mens E-Mail    0.182757    0.012531  1.422617
No E-Mail      0.106167    0.005726  0.652789
Womens E-Mail  0.151400    0.008837  1.077202


Beyond matching group sizes, a randomized assignment should also leave pre-treatment customer characteristics similarly distributed across the three groups; two groups can be the same size and still be unbalanced on who is in them. We check every covariate in the file, numeric and categorical. `visit` is left out on purpose: whether a customer visited the website was recorded after the emails went out, so it is an outcome the emails can change, not a pre-treatment characteristic. It is shown with the other outcomes in the cell above.

In [4]:
# visit, conversion and spend are all measured after the email is sent, so they are outcomes, not covariates
outcome_cols = ("segment", "visit", "conversion", "spend")
covariate_cols = [
    c for c in df.columns
    if c not in outcome_cols and pd.api.types.is_numeric_dtype(df[c])
]
categorical_cols = [
    c for c in df.columns
    if c not in outcome_cols and not pd.api.types.is_numeric_dtype(df[c])
]

print(f"Numeric covariates: {covariate_cols}")
print(df.groupby("segment")[covariate_cols].mean())

for col in categorical_cols:
    print(f"\n{col}, share of each group:")
    print(pd.crosstab(df["segment"], df[col], normalize="index").round(3))

Numeric covariates: ['recency', 'history', 'mens', 'womens', 'newbie']
                recency     history      mens    womens    newbie
segment                                                          
Mens E-Mail    5.773642  242.835931  0.550946  0.551415  0.501525
No E-Mail      5.749695  240.882653  0.553224  0.547639  0.501971
Womens E-Mail  5.767850  242.536633  0.548932  0.550101  0.503250

history_segment, share of each group:
history_segment  1) $0 - $100  2) $100 - $200  3) $200 - $350  4) $350 - $500  \
segment                                                                         
Mens E-Mail             0.363           0.220           0.192           0.098   
No E-Mail               0.357           0.227           0.190           0.100   
Womens E-Mail           0.357           0.221           0.194           0.102   

history_segment  5) $500 - $750  6) $750 - $1,000  7) $1,000 +  
segment                                                         
Mens E-Mail             

Groups are well balanced on size (21,387 Womens, 21,307 Mens, 21,306 No E-Mail) and on every pre-treatment covariate: mean recency (5.75 to 5.77), purchase history ($240.88 to $242.84), the shares who bought men's or women's merchandise, the share of new customers, and the distribution of history segment, zip code type and purchase channel differ by at most about one percentage point between groups. That's consistent with a properly randomized design, so there's nothing to correct for. `visit`, by contrast, differs clearly (18.3% Mens, 15.1% Womens, 10.6% control), which is what you'd expect if the emails bring people to the website: it is an effect of the treatment, not an imbalance.

The raw numbers already suggest a pattern worth testing formally: Mens
E-Mail shows the highest conversion rate (1.25%, more than double the
control's 0.57%) and highest average spend ($1.42 vs. control's $0.65).
Womens E-Mail sits between the two on both metrics (0.88% conversion,
$1.08 spend). Whether these differences are statistically significant, 
and whether they survive Bonferroni correction, is what Sections 2-3 test
formally, rather than concluding anything from these raw averages alone.

## 2. Conversion: Two-Proportion Z-Tests

Conversion is binary, so each treatment group is compared against control
using a two-proportion z-test, the standard test for comparing rates
between two independent groups.

In [5]:
control = df[df["segment"] == "No E-Mail"]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]

results_conversion = []

for name, group in [("Mens", mens), ("Womens", womens)]:
    count = np.array([group["conversion"].sum(), control["conversion"].sum()])
    nobs = np.array([len(group), len(control)])
    z_stat, p_value = proportions_ztest(count, nobs)
    ci_low, ci_high = confint_proportions_2indep(
        count[0], nobs[0], count[1], nobs[1], method="wald", compare="diff"
    )
    results_conversion.append({
        "comparison": f"{name} vs Control",
        "metric": "conversion",
        "group_rate": group["conversion"].mean(),
        "control_rate": control["conversion"].mean(),
        "difference": group["conversion"].mean() - control["conversion"].mean(),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "z_stat": z_stat,
        "p_value": p_value,
    })

results_conversion_df = pd.DataFrame(results_conversion)
results_conversion_df

,comparison,metric,group_rate,control_rate,difference,ci_low,ci_high,z_stat,p_value
0,Mens vs Control,conversion,0.012531,0.005726,0.006805,0.005000,0.008610,7.385114,1.523224e-13
1,Womens vs Control,conversion,0.008837,0.005726,0.003111,0.001499,0.004723,3.779561,1.571051e-04


Both campaigns increase the conversion rate. Mens vs. Control: 1.25% vs.
0.57%, a difference of +0.68 percentage points (95% CI +0.50 to +0.86),
z = 7.39, p ≈ 1.5×10⁻¹³. Womens vs. Control: 0.88% vs. 0.57%, +0.31 points
(95% CI +0.15 to +0.47), z = 3.78, p ≈ 1.6×10⁻⁴. Both p-values are far below
the Bonferroni-adjusted threshold of 0.0125 (0.05/4). The intervals are Wald
intervals for the difference in proportions, not adjusted for the 4 tests.

## 3. Spend: Two-Sample T-Tests

Spend is a continuous dollar amount (zero for non-converters), so each
treatment group is compared against control using an independent two-sample
t-test, Welch's variant, which doesn't assume equal variance between
groups, consistent with the approach used in `05_campaigns_and_coupons.ipynb`. With over 21,000 customers per group, the Central Limit Theorem makes the t-test's normality assumption a non-issue at the level of the *sample mean*, even though `spend` itself is nowhere near normal (a spike at zero for the ~98% who don't convert, a right-skewed tail for those who do); this is the standard justification for using a t-test on revenue-like metrics in large-sample A/B testing, and is worth stating rather than leaving the reader to wonder whether the skew invalidates the test.

In [6]:
results_spend = []

for name, group in [("Mens", mens), ("Womens", womens)]:
    result = stats.ttest_ind(group["spend"], control["spend"], equal_var=False)
    ci = result.confidence_interval(confidence_level=0.95)
    results_spend.append({
        "comparison": f"{name} vs Control",
        "metric": "spend",
        "group_mean": group["spend"].mean(),
        "control_mean": control["spend"].mean(),
        "difference": group["spend"].mean() - control["spend"].mean(),
        "ci_low": ci.low,
        "ci_high": ci.high,
        "t_stat": result.statistic,
        "p_value": result.pvalue,
    })

results_spend_df = pd.DataFrame(results_spend)
results_spend_df

,comparison,metric,group_mean,control_mean,difference,ci_low,ci_high,t_stat,p_value
0,Mens vs Control,spend,1.422617,0.652789,0.769827,0.485140,1.054515,5.300140,1.163815e-07
1,Womens vs Control,spend,1.077202,0.652789,0.424412,0.168957,0.679868,3.256372,1.129397e-03


Both campaigns also increase mean spend per customer, non-buyers included:
Mens +$0.77 (95% CI +$0.49 to +$1.05), t = 5.30, p ≈ 1.2×10⁻⁷; Womens
+$0.42 (95% CI +$0.17 to +$0.68), t = 3.26, p ≈ 1.1×10⁻³. As in Section 2,
Mens shows the stronger effect. Whether all 4 results (2 metrics × 2
comparisons) survive joint correction is tested next.

### 3.1 Decomposing the Spend Effect: More Converters, or Bigger Baskets?

The spend test above answers "does the average dollar amount, across everyone including non-converters, go up," which conflates two different effects: campaigns could lift spend purely by creating more converters (already confirmed in Section 2), by making converters spend more once they convert, or both. We check the second piece directly, restricting the same comparison to customers who converted. This is an exploratory, complementary check, not part of the 4-test family Section 4 corrects for.

In [7]:
mens_converters = mens[mens["conversion"] == 1]
womens_converters = womens[womens["conversion"] == 1]
control_converters = control[control["conversion"] == 1]

for name, group in [("Mens", mens_converters), ("Womens", womens_converters)]:
    result = stats.ttest_ind(group["spend"], control_converters["spend"], equal_var=False)
    ci = result.confidence_interval(confidence_level=0.95)
    print(f"{name} vs Control, spend among converters only: "
          f"${group['spend'].mean():.2f} vs ${control_converters['spend'].mean():.2f} "
          f"(n={len(group)} vs n={len(control_converters)}), t={result.statistic:.3f}, p={result.pvalue:.4f}, "
          f"95% CI of the difference: ${ci.low:.2f} to ${ci.high:.2f}")

Mens vs Control, spend among converters only: $113.53 vs $114.00 (n=267 vs n=122), t=-0.041, p=0.9672, 95% CI of the difference: $-23.25 to $22.30
Womens vs Control, spend among converters only: $121.89 vs $114.00 (n=189 vs n=122), t=0.653, p=0.5145, 95% CI of the difference: $-15.92 to $31.70


Among customers who bought, spend is about the same in every group: $113.53 (Mens) and $121.89 (Womens) against $114.00 in the control group (p = 0.97 and p = 0.51). So the spend lift in Section 3 comes from more customers buying, not from buyers spending more. The intervals are wide, though: with only 122 buyers in the control group, the Mens difference per buyer could be anywhere from −$23 to +$22 and the Womens one from −$16 to +$32, so a moderate change in basket size can't be ruled out. Spend among buyers is also skewed, and with groups this small the t-test leans on the Central Limit Theorem much more than in Section 3.

## 4. Multiple Comparisons Correction

Section 2 (conversion) and Section 3 (spend) produced 4 p-values in total, 2 comparisons × 2 metrics. Bonferroni correction is applied jointly across all 4, not separately within each metric, since all 4 tests belong to the same family of comparisons defined in the introduction.

Bonferroni divides the significance threshold by the number of tests: 0.05 / 4 = 0.0125. Any p-value above this adjusted threshold is not considered significant after correction, even if it would have been at the uncorrected 0.05 level.

In [8]:
all_results = pd.concat([results_conversion_df, results_spend_df], ignore_index=True)

reject, p_adjusted, _, alpha_bonferroni = multipletests(
    all_results["p_value"], alpha=0.05, method="bonferroni"
)

all_results["p_adjusted"] = p_adjusted
all_results["significant_after_correction"] = reject

print(f"Bonferroni-adjusted alpha: {alpha_bonferroni:.4f}")
all_results

Bonferroni-adjusted alpha: 0.0125


,comparison,metric,group_rate,control_rate,difference,ci_low,ci_high,z_stat,p_value,group_mean,control_mean,t_stat,p_adjusted,significant_after_correction
0,Mens vs Control,conversion,0.012531,0.005726,0.006805,0.005000,0.008610,7.385114,1.523224e-13,NaN,NaN,NaN,6.092894e-13,True
1,Womens vs Control,conversion,0.008837,0.005726,0.003111,0.001499,0.004723,3.779561,1.571051e-04,NaN,NaN,NaN,6.284205e-04,True
2,Mens vs Control,spend,NaN,NaN,0.769827,0.485140,1.054515,NaN,1.163815e-07,1.422617,0.652789,5.300140,4.655260e-07,True
3,Womens vs Control,spend,NaN,NaN,0.424412,0.168957,0.679868,NaN,1.129397e-03,1.077202,0.652789,3.256372,4.517588e-03,True


All 4 tests remain significant after Bonferroni correction (adjusted alpha 0.0125). The closest one is Womens vs. Control on spend (p = 0.0011, adjusted p = 0.0045), about ten times below the threshold; the other three are several orders of magnitude below it.

(The `NaN` values are expected, concatenating the two results tables means
each row only populates the columns relevant to its own test type.)

## 5. Business Interpretation

Both campaigns produced a statistically significant lift over control on conversion and spend, and the lift survives Bonferroni correction. Mens E-Mail shows the stronger effect: conversion goes from 0.57% to 1.25% (+0.68 points, 95% CI +0.50 to +0.86) and mean spend per customer from $0.65 to $1.42 (+$0.77, CI +$0.49 to +$1.05). Womens E-Mail adds +0.31 points of conversion (CI +0.15 to +0.47) and +$0.42 of spend (CI +$0.17 to +$0.68). Section 3.1 suggests the spend lift comes mainly from more customers buying, not from larger baskets.

With over 21,000 customers per group, even small differences become statistically detectable, so significance alone says little about size; the confidence intervals do. In absolute terms the effects are small: about 7 extra buyers per 1,000 emails for Mens and 3 for Womens. Mean spend is low because over 98% of customers in every group did not buy; it is an average diluted by zeros, not a typical buyer's spend. A decision to scale the Mens campaign would still need to weigh this lift against the cost of sending it, which this dataset doesn't provide.

Mens E-Mail is the stronger candidate on the evidence here, but the two campaigns weren't compared with each other directly: the 4 tests compare each campaign with the control group only.

In [9]:
print(f"Bonferroni-adjusted alpha: {alpha_bonferroni:.4f}")
all_results[["comparison", "metric", "p_value", "p_adjusted", "significant_after_correction"]]

Bonferroni-adjusted alpha: 0.0125


,comparison,metric,p_value,p_adjusted,significant_after_correction
0,Mens vs Control,conversion,1.523224e-13,6.092894e-13,True
1,Womens vs Control,conversion,1.571051e-04,6.284205e-04,True
2,Mens vs Control,spend,1.163815e-07,4.655260e-07,True
3,Womens vs Control,spend,1.129397e-03,4.517588e-03,True


## 6. Project Conclusion

This microcase closes `P2 — retail-consumer-behavior`. Across 7 notebooks,
the project moved from raw, unverified CSVs to a validated PostgreSQL
database (`01`), a household-level RFM table with documented scoring
choices (`02`), a K-means segmentation that accounts for its sensitivity to
outliers (`03`), a cohort analysis of how steadily panel households stay
active, with a window-edge artifact identified as such (`04`), a campaign
analysis that combines three comparison designs to separate selection from
campaign effects (`05`), and two independent microcases on funnel analysis
(`06`) and a randomized A/B test with multiple-comparisons correction (`07`).

The contrast between `05` and `07` is the main methodological thread. With
retailer-chosen targeting, the campaign analysis had to rule out selection
before saying anything about effects, and in the end found no clear effect
for any single campaign. With randomized assignment, a direct comparison
against control is enough, and both email campaigns show a measurable lift.